# How the migration panel was built: data choices and methods

This notebook documents every decision behind `migration_panel_combined.csv` and `migration_panel_normalized.csv`:
what each column measures, where it comes from, how gaps were filled (or deliberately left blank), and why.
The code cells check each claim against the files themselves, so the numbers here update if the data changes.

Companion files: `migration_panel_codebook.xlsx` (one row per column, with sources and URLs) and
`migration_heatmaps_normalized.ipynb` (the analysis and charts).

**Contents**
1. Research question and scope
2. Unit of observation, timing and normalization
3. Outcome measures (what counts as "immigration")
4. Push-factor indicators
5. Filling gaps: rules and every exception
6. Things deliberately left out
7. Analysis choices
8. Known limitations

In [1]:
import pandas as pd
import numpy as np

comb = pd.read_csv("migration_panel_combined.csv")
norm = pd.read_csv("migration_panel_normalized.csv")
cb = pd.read_excel("migration_panel_codebook.xlsx", sheet_name="Columns")
print("combined:", comb.shape, "| normalized:", norm.shape)
print("countries:", comb.country.nunique(), "| years:", comb.year.min(), "-", comb.year.max())

combined: (2000, 68) | normalized: (2000, 46)
countries: 200 | years: 2014 - 2023


---
# 1. Research question and scope

**Question:** what conditions in a country (push factors) go along with more people from that country coming to the
United States?

**Scope choices**
- **Years: U.S. fiscal years 2014–2023.** FY2024 was in the original panel and was dropped at your request.
- **Countries: 200**, every country or territory in the original panel. Each appears once per year (2,000 rows).
- **Push-factor categories** follow the five categories you set for the project: economic hardship, violence and
  conflict, persecution and discrimination, environmental disasters, and public services. U.S. diaspora is
  included separately as a **pull factor**, since family networks strongly shape who can migrate.

In [2]:
comb.groupby("year").size().to_frame("rows per year").T

year,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
rows per year,200,200,200,200,200,200,200,200,200,200


---
# 2. Unit of observation, timing and normalization

**One row = one origin country in one U.S. fiscal year.**

**Timing: indicators are lagged one year (`_lag1`).** The row for FY *t* pairs immigration in FY *t* with conditions in
calendar year *t−1*. Reasons:
1. Migration responds to conditions with a delay (deciding, saving, applying, travelling).
2. It avoids reverse timing, where the year's emigration itself changes that year's indicators.
3. A U.S. fiscal year (Oct–Sep) straddles two calendar years anyway.

**Normalization: per 100,000 people** in the origin country (`population_lag1`, World Bank). Raw counts mostly
reflect country size: India and Mexico top every raw list. Dividing by population asks whether a *larger share* of a
country's people leave, which is what a push factor should predict.
- The **normalized file** keeps only per-capita measures (per 100k, %, indexes, per-capita dollars) so it can feed the
  correlation charts directly. The **combined file** also has the raw counts.
- Indicators that are already rates (%, per 1,000 births, V-Dem scores, GDP per capita) were not divided again.

In [3]:
ex = comb[(comb.country == "El Salvador") & (comb.year == 2019)]
ex[["country", "year", "refugees", "asylum_grants", "population_lag1", "asylum_grants_per_100k"]].assign(
    check=lambda d: (d.asylum_grants / d.population_lag1 * 1e5).round(2))

,country,year,refugees,asylum_grants,population_lag1,asylum_grants_per_100k,check
575,El Salvador,2019,310.0,860.0,6219807.0,13.826796,13.83


---
# 3. Outcome measures: what counts as "immigration"

The project focuses on **people coming to the United States**. The outcomes differ in whether they capture the moment
of entry or a later change of legal status, so they're treated differently in the analysis.

| Column | Measures | Entry? | Source |
|---|---|---|---|
| `refugees` | Refugees resettled from abroad, by nationality | **Yes** | DHS Yearbook Table 14 |
| `apprehensions` | Noncitizen apprehensions (attempted unauthorized entry), by nationality | **Mostly** | DHS Yearbook 2022 Table 34; FY2023 CBP |
| `parole_major_programs` | People paroled in under CHNV / Uniting for Ukraine | **Yes** | DHS FY2023 parole report |
| `green_cards` | New lawful permanent residents, by country of birth | **Partly** | DHS Yearbook Table 3 |
| `green_cards_excl_refugee_asylee` | Same, minus refugee/asylee adjustments | **Partly** | DHS Tables 3 and 10 |
| `green_cards_<type>` | Green cards split into six classes of admission | **Partly** | DHS Yearbook Table 10 |
| `asylum_grants` | Affirmative asylum grants (USCIS), by nationality | **No** | DHS Yearbook Table 17 |

### Choices and why

**Green cards include adjustments of status.** `green_cards` counts each person once, in the year they first become a
permanent resident. Many were already in the U.S. (students, workers, spouses on other visas) and switched status.
They are kept because the project tracks *immigration to the U.S.* broadly: someone who arrives on a student visa
and stays is an immigrant. Push factors can also drive the decision to stay. In the write-up this column is
described as **"new permanent residents,"** not "arrivals."

**For analysis, green cards exclude the refugee/asylee class** (`green_cards_excl_refugee_asylee`). That class is:
- people already counted in `refugees` or `asylum_grants` in an earlier year (refugees must apply after one year),
  so including it would count them twice; and
- **Cuban Adjustment Act** green cards, which DHS files under "Refugees" (detailed class CU6, Yearbook Table 7). In
  FY2022, classes CU6 and CU7 were 15,482 of the 29,423 refugee adjustments. Including them made Cuba look like a
  refugee-driven green card country.

**Green card types** (Table 10): immediate relatives of U.S. citizens, family preferences, employment, diversity
lottery, refugees and asylees, and other. The analysis uses the five classes *other than* refugees and asylees.

**Asylum grants are treated as secondary.** Affirmative applicants are already in the U.S., often having entered on a
visa years before, so the grant year says little about when they came. Defensive asylum (granted by immigration
judges to people in deportation proceedings) is not in the dataset. Those applicants mostly entered at the border,
but court backlogs mean the grant usually comes years later, so it is also a poor measure of entries.

**Apprehensions are the unauthorized-entry measure.** They count *events*, not people: someone caught twice counts
twice. DHS's Table 34 also includes some ICE arrests inside the country, so it isn't purely a border measure.

**Parole is recorded but not analyzed.** The country-specific programs (CHNV, Uniting for Ukraine) are clean entry
counts, but DHS publishes them by country for FY2023 only and they cover five countries, so there's no variation
across country-years to correlate with.

In [4]:
outcome_cols = ["refugees", "apprehensions", "parole_major_programs", "green_cards",
                "green_cards_excl_refugee_asylee", "asylum_grants"]
cov = comb.groupby("year")[outcome_cols].apply(lambda d: d.notna().sum())
cov.T  # number of countries with a value, by year

year,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
refugees,200,200,200,200,200,200,200,200,200,200
apprehensions,195,195,195,197,197,195,191,194,197,12
parole_major_programs,200,200,200,200,200,200,200,199,198,199
green_cards,200,200,200,200,200,200,200,200,200,200
green_cards_excl_refugee_asylee,195,196,200,0,196,195,196,197,197,195
asylum_grants,200,200,200,200,200,200,200,200,200,200


In [5]:
# Cuba: why the refugee/asylee class is removed
comb[comb.country == "Cuba"][["year", "green_cards", "green_cards_refugee_asylee", "green_cards_excl_refugee_asylee",
                               "refugees"]].set_index("year")

,green_cards,green_cards_refugee_asylee,green_cards_excl_refugee_asylee,refugees
year,,,,
2014,46680.0,40086.0,6594.0,4060.0
2015,54400.0,47803.0,6597.0,1530.0
2016,66520.0,57036.0,9484.0,350.0
2017,65030.0,NaN,NaN,180.0
2018,76490.0,69860.0,6630.0,0.0
2019,41640.0,32900.0,8740.0,0.0
2020,16370.0,12148.0,4222.0,10.0
2021,23080.0,19354.0,3726.0,0.0
2022,36640.0,14548.0,22092.0,0.0


---
# 4. Push-factor indicators

All are lagged one year. Full source details and URLs are in the codebook's *Columns* and *Sources* sheets.

| Category | Indicators | Source |
|---|---|---|
| Economic hardship | GDP per capita (constant 2015 US$), GDP growth, unemployment, youth unemployment, inflation | World Bank WDI |
| Violence & conflict | Conflict IDPs, new conflict displacements | IDMC (file you supplied) |
| | Battle deaths | World Bank WDI (UCDP data) |
| | Homicides per 100k | UNODC via WDI; gaps from IHME GBD 2023 |
| | Political violence | V-Dem v16 |
| Persecution & discrimination | Electoral democracy, freedom from killings/torture, religious freedom, power equality by social group / gender / sexual orientation | V-Dem v16 |
| Environmental disasters | Disaster deaths, people affected, disaster count | Original panel (likely EM-DAT; **not verified**) |
| | Disaster IDPs, new disaster displacements | IDMC |
| | Undernourishment, food production index | World Bank WDI (FAO data) |
| Public services | Life expectancy, under-5 mortality, basic water access, electricity access, health spending per capita | World Bank WDI |
| Pull factor | U.S. diaspora (people born in the country living in the U.S., 2023) | Original panel (likely Census ACS; cross-checked against UN DESA) |

### Choices and why
- **Proxies.** None of the five categories can be measured directly, so each uses several standard proxies. V-Dem's
  expert-coded scores are the best available cross-country measure of persecution. They're latent scales (roughly −4 to +4),
  where higher means more freedom, so a *negative* correlation with immigration means repression goes with more migration.
- **World Bank data** came through the open-numbers GitHub mirror of WDI (the World Bank site wasn't reachable from the
  build environment). Values were checked to match WDI exactly.
- **Removed at your request:** poverty headcount and primary school completion.
- **Diaspora is a single 2023 value** per country. It can explain differences *between* countries but not changes over time.

In [6]:
cats = cb[~cb.column.str.endswith(("_source", "_method"))].groupby("category").column.apply(list)
for k, v in cats.items():
    if k not in ("ID",):
        print(f"{k} ({len(v)}): {', '.join(v)}\n")

Economic hardship (5): gdp_per_capita_lag1, unemployment_pct_lag1, gdp_growth_pct_lag1, inflation_pct_lag1, youth_unemployment_pct_lag1

Environmental disasters (11): disaster_deaths_lag1, disaster_affected_lag1, disaster_affected_per_100k_lag1, disaster_count_lag1, idps_disaster_lag1, new_displacements_disaster_lag1, new_displacements_disaster_per_100k_lag1, undernourishment_pct_lag1, food_production_index_lag1, disaster_deaths_per_100k_lag1, idps_disaster_per_100k_lag1

Internal displacement (2): idps_total_lag1, idps_total_per_100k_lag1

Other (1): population_lag1

Outcome (24): green_cards, refugees, asylum_grants, green_cards_per_100k, green_cards_excl_refugee_asylee, green_cards_excl_refugee_asylee_per_100k, green_cards_family, green_cards_employment, green_cards_immediate_relatives, green_cards_diversity, green_cards_refugee_asylee, green_cards_other, green_cards_family_per_100k, green_cards_employment_per_100k, green_cards_immediate_relatives_per_100k, green_cards_diversity_per

---
# 5. Filling gaps: rules and every exception

**General rule: a blank is only filled from a published source, and every filled value is labelled.** No values are
interpolated, averaged from neighbours or modelled, with one bounded exception (the green card midpoint below, which
is flagged row by row). When no source exists, the cell stays blank and the reason is in the codebook.

### Blank means zero (where the source's blank means "none")
- **Refugees and asylum grants:** countries DHS doesn't list in a year sent none (or so few they fall into DHS's
  "all other" line), so they're **0**.
- **Apprehensions 2014–2022:** countries DHS doesn't list separately are **0** (all such countries together total
  13–28 a year). Hong Kong, Macau and the United States are left blank.
- **Battle deaths:** blank in UCDP/WDI means no recorded state-based conflict → **0**.
- **IDMC new displacements:** blank → **0**.
- **IDMC conflict IDPs:** blank → **0**, *except* where IDMC recorded new displacements that year but published no
  year-end total (e.g. El Salvador 2016–2021). Those are left **blank**, because 0 would be wrong.
- **Green card classes:** countries not listed in Table 10 that year *and* with 0 total green cards → **0**.

### Gaps filled from other sources (each labelled in a `_source` column)
- **Homicides:** UNODC first; remaining gaps filled with IHME GBD 2023 "interpersonal violence" estimates (rank
  correlation with UNODC 0.91). See `homicides_source`.
- **Inflation:** Eritrea 2013–2018 and Turkmenistan 2013–2022 from IMF Regional Economic Outlook series on FRED; Somalia
  from World Bank figures as republished by two independent sites (marked secondary). See `inflation_source`.
- **Kosovo and Curaçao:** the WDI mirror codes them `kos` and `nld_curacao`; once matched, 126 blank cells were filled
  with real WDI values.
- **Refugees FY2014 and Kuwait:** from DHS Yearbook 2023 Table 14.
- **Apprehensions FY2023:** the DHS table wasn't reachable, so only 12 countries are filled, with CBP **Border Patrol**
  apprehensions + expulsions. These are the countries where that narrower count was close to DHS's in 2022. Port-of-entry
  counts were *not* used because they include CBP One and CHNV parole arrivals, which are lawful entries. See
  `apprehensions_source`.

### Green card types (Table 10) and the excl. refugee/asylee column
- **FY2017 is blank**: the DHS page for that year couldn't be read.
- **"D" cells** (suppressed by DHS for privacy) are blank. DHS always hides a second cell in the same row, so a hidden
  value is bounded: between 0 and the row's gap (total minus the published classes).
- **`green_cards_excl_refugee_asylee`:** exact where the refugee/asylee cell is published. Where it's suppressed, the
  value is the midpoint of those bounds, used only when the gap is ≤ 30 people or ≤ 5% of the row. The maximum
  possible error is written in `green_cards_excl_ra_method`.
- **FY2023 values are rounded to the nearest 10 by DHS**, so classes may not sum exactly to the total.
- **Mali and Malta FY2023:** the Table 10 page has their labels swapped. They were swapped back because DHS Table 3 for
  FY2023 (Mali 540, Malta 40) and every earlier year agree with that.

### Verification
Every DHS table extracted for this project was checked by summing countries to DHS's published totals (and, for Table
10, each country's classes to its total). Any gaps came only from suppressed cells or rounding.

In [7]:
print(comb.homicides_source.value_counts(dropna=False).to_string(), "\n")
print(comb.inflation_source.value_counts(dropna=False).to_string(), "\n")
print(comb.apprehensions_source.value_counts(dropna=False).to_string(), "\n")
m = comb.green_cards_excl_ra_method.fillna("blank").str.replace(r", max error.*", "", regex=True)
print(m.value_counts().to_string())

homicides_source
UNODC via World Bank WDI       1213
IHME GBD 2023 via Gapminder     704
NaN                              83 

inflation_source
World Bank WDI (FP.CPI.TOTL.ZG)                                                         1784
NaN                                                                                      190
World Bank FP.CPI.TOTL.ZG via historysaid.com / statisticsoftheworld.com (secondary)      10
IMF Regional Economic Outlook (MCD) via FRED TRNPCPIPCHPT                                 10
IMF Regional Economic Outlook (SSA) via FRED ERIPCPIPCPPPT                                 6 

apprehensions_source
DHS Yearbook 2022 Table 34                                                                       1756
NaN                                                                                               232
CBP Border Patrol encounters only, FY2023 (narrower: excludes ports of entry and ICE arrests)      12 

green_cards_excl_ra_method
exact (green_cards - refugee/asyl

In [8]:
# Green card check: the six classes should add up to green_cards except where a cell is suppressed or rounded
T = ["green_cards_family", "green_cards_employment", "green_cards_immediate_relatives",
     "green_cards_diversity", "green_cards_refugee_asylee", "green_cards_other"]
s = comb[T].sum(axis=1, min_count=1)
gap = (comb.green_cards - s)[s.notna()]
has_D = comb.loc[s.notna(), T].isna().any(axis=1)
print("rows with class data:", int(s.notna().sum()))
print("gap > 50 people:", int((gap.abs() > 50).sum()), "| of those, rows with a suppressed cell:", int(has_D[gap.abs() > 50].sum()))

rows with class data: 1770
gap > 50 people: 49 | of those, rows with a suppressed cell: 49


### Gaps left blank on purpose (no published source)
- **Diaspora** for countries the Census and UN only report inside regional "Other" groups.
- **GDP per capita** for Eritrea (World Bank series ends 2011) and North Korea (never published).
- **Inflation** for Cuba and North Korea.
- **Undernourishment** where FAO publishes no estimate (e.g. Yemen, Eritrea, Burundi).
- **V-Dem** for 28 small states it doesn't cover.
- **Disaster IDPs and total IDPs** before 2020: IDMC only started counting disaster IDP stocks in 2019.
- **Apprehensions FY2023** outside the 12 countries; **parole** for Afghanistan 2021–2023 and Ukraine 2022.

In [9]:
miss = (norm.isna().mean() * 100).round(1).sort_values(ascending=False)
miss[miss > 0].to_frame("% of rows blank")

,% of rows blank
idps_total_per_100k_lag1,63.3
idps_disaster_per_100k_lag1,62.8
diaspora_in_us_per_100k_origin_pop,38.0
green_cards_other_per_100k,28.1
green_cards_refugee_asylee_per_100k,24.2
green_cards_diversity_per_100k,23.5
green_cards_family_per_100k,22.0
green_cards_employment_per_100k,21.6
undernourishment_pct_lag1,19.1
green_cards_immediate_relatives_per_100k,14.4


---
# 6. Things deliberately left out

- **Temporary Protected Status (TPS).** TPS is granted to people *already in the U.S.* when their home country has a
  crisis. It isn't a way of entering. Not added, per your decision.
- **Compacts of Free Association (COFA).** Citizens of the Marshall Islands, Micronesia and Palau can live and work in the
  U.S. without a visa, so they rarely appear in any of these measures even though large shares live in the U.S.
  They stay in the data with a footnote. Treat them as outliers in green card analysis.
- **Defensive asylum** (immigration court grants): not added. See section 3 for why grants are a weak entry measure.
- **Poverty headcount and primary completion:** removed at your request.

---
# 7. Analysis choices (used in the heatmap notebook)

- **Spearman rank correlation**, not Pearson. Immigration rates and many indicators are extremely skewed (a few countries
  have rates hundreds of times the median), and Spearman uses ranks, so outliers don't dominate.
- **Pooled country-years** (all countries and years together). Correlations mostly reflect differences *between*
  countries, plus some over time.
- **Pairwise deletion:** each correlation uses every row where *both* the outcome and that indicator are present. A blank
  in one indicator doesn't drop the row from the others.
- **"Strong" = |ρ| ≥ 0.40.** With about 2,000 rows almost every correlation is statistically significant (above ±0.2,
  p < 10⁻¹²), so significance can't separate meaningful relationships from trivial ones. Instead the cutoff uses effect size.
- **Color scale fixed at ±0.5** so charts are comparable.
- **Correlation is not causation.** Many indicators move together (poor countries also have lower life expectancy and
  less democracy), so a strong correlation for one indicator may partly reflect others.

---
# 8. Known limitations

1. **Disaster deaths / affected / count** come from the original panel and could not be checked (EM-DAT requires a login).
2. **Diaspora** comes from the original panel and is a single 2023 value. It was cross-checked against UN DESA (within
   about 9%, rank correlation 0.98) but not traced to the exact Census table.
3. **Green cards mix new arrivals and adjustments of status**, and their yearly numbers depend heavily on U.S. visa caps,
   backlogs and processing speed (e.g. the 2020–21 dip was COVID, not conditions abroad).
4. **FY2017 green card types are missing**, so `green_cards_excl_refugee_asylee` has no 2017.
5. **Apprehensions count events, not people**, and FY2023 covers only 12 countries with a narrower measure.
6. **Parole** is a one-year snapshot for five countries.
7. **Measures overlap across years**: refugees and asylees later become green card holders; parolees and people
   apprehended may later get asylum. Never add the outcome columns together.
8. **Small countries swing wildly per capita**: a handful of people from a country of 100,000 is a large rate.

**Sources:** see the *Sources* sheet of `migration_panel_codebook.xlsx` for full citations and URLs.

In [10]:
pd.read_excel("migration_panel_codebook.xlsx", sheet_name="Sources")

,source,url,used_for
0,"DHS Office of Homeland Security Statistics, Ye...",https://ohss.dhs.gov/topics/immigration/yearbook,Original panel outcome columns (traced and spo...
1,"DHS Office of Homeland Security Statistics, Ye...",https://ohss.dhs.gov/topics/immigration/yearbo...,apprehensions (unauthorized-immigration proxy)
2,"DHS Office of Homeland Security Statistics, Ye...",https://ohss.dhs.gov/topics/immigration/yearbo...,Green cards by class
3,"World Bank, World Development Indicators (acce...",https://databank.worldbank.org/source/world-de...,"Economic, conflict, food and public-service in..."
4,"U.S. Customs and Border Protection, Nationwide...",https://www.cbp.gov/document/stats/nationwide-...,"FY2023 apprehensions (Border Patrol component,..."
5,"DHS, Parole Requests - Fiscal Year 2023, Fourt...",https://dhs.gov/sites/default/files/2024-07/20...,parole_major_programs
6,IMF Regional Economic Outlook series on FRED: ...,https://fred.stlouisfed.org/data/ERIPCPIPCPPPT...,Inflation gaps
7,"World Bank FP.CPI.TOTL.ZG for Somalia, as repu...",https://historysaid.com/somalia-fed-rep/inflat...,Somalia inflation
8,"Coppedge, M. et al. (2026). V-Dem Country-Year...",https://www.v-dem.net ; https://github.com/vde...,Persecution and political-violence indicators
9,Internal Displacement Monitoring Centre (2026)...,https://www.internal-displacement.org/database...,Internal displacement (file you supplied)
